**Machine Learning**

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Modeling
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Reproducibility
# 42 started as a joke from Douglas Adam's Book: Hitchhiker's Guide to the Galaxy. It was later adopted due to its mass popularity
RANDOM_STATE = 42 

# Data: use scikit-learn California Housing (clean, numeric)
from sklearn.datasets import fetch_california_housing
data_bunch = fetch_california_housing(as_frame=True)
df = data_bunch.frame.copy()
df.head()


Supervised learning maps inputs X to outputs y by minimizing a loss; trees partition feature space with rules to predict continuous values for regression or classes for classification, which aligns with beginner-friendly introductions used in many ML curricula.

In [ ]:
# Feature matrix X and target y
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Simple decision tree model
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X, y)

# Intuition: model learns mappings from features to target by minimizing error on training data.
print(f"Training R^2: {tree.score(X, y):.3f}")


Basic EDA includes summary statistics, missingness checks, and correlations to understand distributions and relationships; this mirrors the “Basic Data Exploration” step emphasized in beginner guides.

In [ ]:
display(df.describe())

# Quick missing check
missing = df.isna().sum().sort_values(ascending=False)
display(missing)

# Correlations
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), cmap="vlag", center=0)
plt.title("Feature correlations")
plt.show()

# Simple pairplot sample
sns.pairplot(df.sample(500, random_state=RANDOM_STATE), diag_kind="hist")
plt.show()


A train/validation split gives a quick performance estimate; decision trees are a friendly first model due to low preprocessing needs and interpretability. 

In [ ]:
# Train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

baseline_tree = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=5)
baseline_tree.fit(X_train, y_train)

preds = baseline_tree.predict(X_valid)
mae = mean_absolute_error(y_valid, preds)
r2  = r2_score(y_valid, preds)

print(f"Validation MAE: {mae:.3f}")
print(f"Validation R^2: {r2:.3f}")


In [ ]:
# k-fold cross-validation with MAE (negative in sklearn, so flip sign)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=5)

cv_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")
print(f"CV MAE mean: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# Compare alternative depth settings
depths = [3, 5, 10, None]
results = []
for d in depths:
    mdl = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=d)
    scores = -cross_val_score(mdl, X, y, cv=kf, scoring="neg_mean_absolute_error")
    results.append({"max_depth": d if d else "None", "mae_mean": scores.mean()})

pd.DataFrame(results)

Shallow trees may underfit (high bias) while very deep trees can overfit (high variance); the bias-variance tradeoff frames this balance and is foundational to preventing under/overfitting in practice.

In [ ]:
train_scores, valid_scores = [], []
depth_grid = list(range(1, 21))

for d in depth_grid:
    mdl = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=d)
    mdl.fit(X_train, y_train)
    train_scores.append(mean_absolute_error(y_train, mdl.predict(X_train)))
    valid_scores.append(mean_absolute_error(y_valid, mdl.predict(X_valid)))

plt.figure(figsize=(7,5))
plt.plot(depth_grid, train_scores, label="Train MAE")
plt.plot(depth_grid, valid_scores, label="Validation MAE")
plt.xlabel("max_depth")
plt.ylabel("MAE (lower is better)")
plt.legend()
plt.title("Bias-variance via depth sweep")
plt.show()


Random forests build many trees on bootstrapped samples and random feature subsets, then aggregate predictions (mean for regression, mode for classification), which often improves generalization and reduces overfitting relative to a single tree.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_valid)
rf_mae = mean_absolute_error(y_valid, rf_preds)
rf_r2  = r2_score(y_valid, rf_preds)
print(f"Random Forest - Validation MAE: {rf_mae:.3f}, R^2: {rf_r2:.3f}")

# Feature importance
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(imp)

plt.figure(figsize=(7,4))
sns.barplot(x=imp.values, y=imp.index, orient="h")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()
